# Random Forest Baseline for IoT Traffic Classification

**Goal.** Train a Random Forest that labels a network-traffic record as **benign** (`0`) or **malicious** (`1`),
and measure how well it does on data it has not seen.

### What is a Random Forest?

A *decision tree* classifies a record by asking a chain of yes/no questions about its features, such as
"is this packet count above some value?". One tree can overfit the training data. A **Random Forest**
trains many trees, each on a random resample of the rows and a random subset of features at every split,
then lets them vote. Averaging many different trees gives steadier predictions than any single tree.

### How this notebook works

1. Set up the environment and choose settings.
2. Download and verify the dataset.
3. Train on a seeded, class-balanced sample of the TRAIN split.
4. Evaluate on the VALIDATION split and save everything under `outputs/`.

The dataset has **115 numeric features** per record, taken from N-BaIoT network traffic.
The model is a *baseline*: it uses mostly default settings so later, tuned models have something to be
compared against.

## 1. Set up the environment

Everything runs on your own machine. The notebook needs a few Python libraries, all listed in
`requirements.txt` next to this file. The cell below installs them into the environment the notebook
is using. If you prefer a virtual environment, create and activate one first, then select it as the
notebook kernel:

```
python -m venv .venv
.venv\Scripts\activate        # Windows
source .venv/bin/activate      # macOS / Linux
```

Run the install cell once. Afterwards you can skip it.

In [ ]:
%pip install -q -r requirements.txt

## 2. Choose the experiment settings

All settings live in this one cell so a run can be repeated exactly.

**Dataset package.** The processed N-BaIoT package is stored on Google Drive. `drive_url` is the share
link, `package_dir` is where the package is kept on your computer, and `expected_manifest_sha256` is
the fingerprint of the package's manifest. The notebook checks this fingerprint so it only ever trains
on the agreed version of the data.

**Run settings.**

- `FRACTION` is the share of the TRAIN split used for training. Rows are drawn separately from each
  class, so benign and malicious traffic keep their proportions. A value of `1` uses every TRAIN row.
  The default `0.05` is a good size for a laptop.
- `JOBS` is the number of CPU threads to use. Lower it if your machine becomes unresponsive.
- `CHUNK_SIZE` is how many VALIDATION rows are scored at a time, which keeps memory use small.
- `TAG` is an optional short label added to the run name, and `OVERWRITE` allows a finished run with
  the same name to be replaced.

**Random Forest settings.**

- `TREES` is the number of trees in the forest. More trees are steadier but slower.
- `MAX_DEPTH` limits how many questions a tree may ask in a row. `None` means no limit.
- `MIN_SAMPLES_LEAF` is the fewest rows allowed in a final leaf. Larger values make smoother trees.
- `MAX_FEATURES` is how many features each split may consider: `"sqrt"`, `"log2"`, a fraction such as
  `0.3`, or a whole number.

**Seeds.** A seed fixes the random choices so the same settings always give the same result.

In [ ]:
CONFIG = {
    "drive_url": "https://drive.google.com/file/d/1D6sXnp8l7-7YwBaonKoTZKcPG1ULKAuL/view?usp=drive_link",
    "package_dir": "../g3b_primary_v1",
    "expected_manifest_sha256": "6ab99e86397242e219bfa1f8490d8e03b169ec87a092af80cb3e7aef1e61147b",
    "output_root": "outputs",
}

SEEDS = {"python": 42, "numpy": 42, "scikit_learn": 42}

FRACTION = 0.05       # share of TRAIN rows used for training, in (0, 1]
JOBS = 2              # CPU threads
CHUNK_SIZE = 50_000   # VALIDATION rows scored per step
TAG = ""              # optional label, letters / digits / . _ -
OVERWRITE = False     # allow replacing a finished run with the same name

TREES = 100
MAX_DEPTH = None
MIN_SAMPLES_LEAF = 1
MAX_FEATURES = "sqrt"   # "sqrt", "log2", a fraction in (0, 1], or a whole number

## 3. Import the libraries

`numpy` handles numeric arrays, `pandas` builds readable tables, and `matplotlib` draws charts.
`prepare_package` and `matching_manifest` come from `dataset_source.py`, which sits beside this notebook
and takes care of downloading the dataset from Google Drive.

In [ ]:
import hashlib
import json
import platform
import random
import re
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from IPython.display import display

from dataset_source import matching_manifest, prepare_package

## 4. Check the settings and name the run

Before doing any work the notebook checks that the settings make sense, fixes the random seeds, and
builds a **run name** such as `rf_frac0.05_seed42`. The run name becomes the name of the output folder.
If a finished run with that name already exists, the notebook stops rather than silently overwriting it.

In [ ]:
if not 0 < FRACTION <= 1:
    raise ValueError("FRACTION must be in (0, 1]")
if TAG and not re.fullmatch(r"[A-Za-z0-9._-]+", TAG):
    raise ValueError("TAG may contain only letters, digits, dot, dash and underscore")
if MAX_DEPTH is not None and MAX_DEPTH < 1:
    raise ValueError("MAX_DEPTH must be at least 1 or None")
if MIN_SAMPLES_LEAF < 1:
    raise ValueError("MIN_SAMPLES_LEAF must be at least 1")
if isinstance(MAX_FEATURES, str):
    if MAX_FEATURES not in ("sqrt", "log2"):
        raise ValueError('MAX_FEATURES text must be "sqrt" or "log2"')
elif isinstance(MAX_FEATURES, int):
    if MAX_FEATURES < 1:
        raise ValueError("A MAX_FEATURES count must be at least 1")
elif not 0 < MAX_FEATURES <= 1:
    raise ValueError("A MAX_FEATURES fraction must be in (0, 1]")

random.seed(SEEDS["python"])
np.random.seed(SEEDS["numpy"])
SEED_SAMPLE = SEEDS["numpy"]      # controls which training rows are drawn
SEED_MODEL = SEEDS["scikit_learn"]  # controls the model's own randomness

RUN_ID = f"rf_frac{FRACTION:g}_seed{SEED_MODEL}" + (f"_{TAG}" if TAG else "")
RUN_DIR = Path(CONFIG["output_root"]) / RUN_ID
if (RUN_DIR / "result.json").exists() and not OVERWRITE:
    raise FileExistsError(f"{RUN_DIR} already holds a finished run. Change TAG or set OVERWRITE = True.")
print("Run name:", RUN_ID)
print("Output folder:", RUN_DIR)

## 5. Get the dataset

`prepare_package` looks for the dataset in `package_dir`. If it is already there, it is reused. If not,
it downloads the package from Google Drive, unpacks it, and checks the manifest fingerprint before
saving it. The first download is several GiB, so watch this cell for progress. Later runs skip the
download.

In [ ]:
PACKAGE_DIR = prepare_package(CONFIG, CONFIG["drive_url"])

## 6. Verify and open the package

Training on a corrupted or wrong version of the data would make every result meaningless, so this
step confirms the package is the expected one:

1. The manifest's fingerprint must match `expected_manifest_sha256`.
2. Every TRAIN and VALIDATION file is hashed and compared with the checksum recorded in the manifest.
3. The package's own loader opens the data as read-only arrays on disk (memory-mapped), so the
   multi-GiB arrays are not loaded into RAM all at once.

The **TEST** split is deliberately never opened. It stays sealed for the final evaluation, so the
model is only ever judged on TRAIN and VALIDATION here.

In [ ]:
manifest_path = matching_manifest(PACKAGE_DIR, CONFIG["expected_manifest_sha256"])
sys.path.insert(0, str(PACKAGE_DIR.resolve()))
from primary_package import PrimaryPackage, sha256_file

package = PrimaryPackage(PACKAGE_DIR, verify=False)
manifest = package.manifest

checked = {}
for name, info in manifest["files"].items():
    if name.startswith("TEST_"):
        continue                      # TEST stays sealed
    if sha256_file(package.root / name) != info["sha256"]:
        raise ValueError(f"Package checksum mismatch: {name}")
    checked[name] = info

package_verification = {
    "status": "PASS",
    "manifest_sha256": sha256_file(manifest_path),
    "checked_payloads": checked,
    "partitions": ["TRAIN", "VALIDATION"],
    "TEST_opened": False,
}

x_train, y_train, train_metadata = package.load("TRAIN")
print("Package version:", manifest["package_version"])
display(pd.DataFrame(
    {"rows": [manifest["counts"]["TRAIN"], manifest["counts"]["VALIDATION"]], "features": [115, 115]},
    index=["TRAIN", "VALIDATION"],
))

## 7. Define helper functions

These small functions are used in the steps that follow.

- `stratified_indices` picks the same share of rows from each class, using a seeded random generator.
- `selected_population` fingerprints exactly which TRAIN rows were used, so the sample can be audited later.
- `binary_metrics` turns the model's predicted probabilities into the confusion matrix and the scores
  reported for the run. A row is called **malicious** when its probability is at least `0.5`.
  **Malicious** is the positive class, so a *false positive* is benign traffic flagged as malicious.
  A rate that cannot be computed (division by zero) is stored as `null` instead of an invented zero.
- `gather_float32` copies the chosen rows into memory as 32-bit numbers, one block at a time.
- `predict_validation` scores VALIDATION rows in chunks and returns the labels with the model's
  probabilities.
- `peak_memory_mb` records how much memory the process used, when `psutil` is available.

In [ ]:
def stratified_indices(y, fraction, seed):
    """Draw the same fraction of rows from each class, returned in sorted order."""
    rng = np.random.default_rng(seed)
    parts = []
    for cls in (0, 1):
        idx = np.flatnonzero(y == cls)
        n = max(1, round(len(idx) * fraction))
        parts.append(rng.choice(idx, size=n, replace=False))
    return np.sort(np.concatenate(parts))


def selected_population(metadata, indices, chunk_size=100_000):
    """Fingerprint the exact TRAIN rows that were selected."""
    idx = np.asarray(indices)
    if (idx.ndim != 1 or not len(idx) or idx.dtype.kind not in "iu"
            or idx[0] < 0 or idx[-1] >= len(metadata) or np.any(idx[1:] <= idx[:-1])):
        raise ValueError("TRAIN indices must be sorted, unique, integral and in range")

    class _HashSink:
        def __init__(self):
            self.digest = hashlib.sha256()
        def write(self, data):
            self.digest.update(data)
            return len(data)

    sink = _HashSink()
    np.lib.format.write_array_header_1_0(sink, {
        "descr": np.lib.format.dtype_to_descr(metadata.dtype),
        "fortran_order": False,
        "shape": (len(idx),),
    })
    for start in range(0, len(idx), chunk_size):
        sink.write(np.asarray(metadata[idx[start:start + chunk_size]]).tobytes(order="C"))
    return {
        "kind": "full" if len(idx) == len(metadata) else "subset",
        "rows": len(idx),
        "rows_in_train": len(metadata),
        "indices_sha256": hashlib.sha256(idx.astype("<i8").tobytes()).hexdigest(),
        "indices_representation": "sorted unique little-endian int64 bytes",
        "train_population_sha256": sink.digest.hexdigest(),
        "population_representation": "sha256 of NumPy v1 selected metadata array",
    }


def binary_metrics(y_true, probabilities, threshold=0.5):
    """Confusion matrix and per-class scores for VALIDATION. Malicious is the positive class."""
    y = np.asarray(y_true)
    p = np.asarray(probabilities, dtype=np.float64)
    if (y.ndim != 1 or p.shape != y.shape or not len(y) or not np.isin(y, [0, 1]).all()
            or not np.isfinite(p).all() or ((p < 0) | (p > 1)).any()):
        raise ValueError("Need non-empty, aligned binary targets and probabilities in [0, 1]")
    pred = p >= threshold
    tn = int(((y == 0) & ~pred).sum())
    fp = int(((y == 0) & pred).sum())
    fn = int(((y == 1) & ~pred).sum())
    tp = int(((y == 1) & pred).sum())

    def rate(a, b):
        return a / b if b else None

    per_class = {}
    for label, a, b, c in (("benign", tn, fn, fp), ("malicious", tp, fp, fn)):
        per_class[label] = {"precision": rate(a, a + b), "recall": rate(a, a + c),
                            "f1": rate(2 * a, 2 * a + b + c), "support": a + c}
    f1s = [v["f1"] for v in per_class.values()]
    return {
        "partition": "VALIDATION", "threshold": threshold, "positive_class": "malicious",
        "confusion_order": ["benign", "malicious"],
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "accuracy": (tn + tp) / len(y), "per_class": per_class,
        "macro_f1": sum(f1s) / 2 if all(v is not None for v in f1s) else None,
        "fpr": rate(fp, fp + tn),
    }


def peak_memory_mb():
    try:
        import psutil
        info = psutil.Process().memory_info()
        to_mb = lambda b: round(b / 2**20, 1)
        return {
            "peak_working_set_mb": to_mb(getattr(info, "peak_wset", info.rss)),
            "peak_private_mb": to_mb(info.peak_pagefile) if hasattr(info, "peak_pagefile") else None,
        }
    except Exception:
        return None


def gather_float32(x, idx, block=50_000):
    """Copy the selected rows into memory as float32, one block at a time."""
    out = np.empty((len(idx), x.shape[1]), dtype=np.float32)
    for start in range(0, len(idx), block):
        out[start:start + block] = x[idx[start:start + block]]
    return out


def predict_validation(model, package, chunk_size):
    """Score every VALIDATION row in chunks; return labels and malicious probabilities."""
    ys, ps = [], []
    for block, y_block in package.chunks("VALIDATION", scaled=False, chunk_size=chunk_size):
        ps.append(model.predict_proba(block.astype(np.float32))[:, 1])
        ys.append(np.asarray(y_block))
    return np.concatenate(ys), np.concatenate(ps)

## 8. Choose the training rows

The full TRAIN split holds millions of rows. Using a seeded, class-balanced sample keeps training
quick while still representing both kinds of traffic. Only TRAIN rows are ever drawn, so no information
from VALIDATION leaks into training. If `FRACTION = 1`, every TRAIN row is used.

The notebook also records a fingerprint of the chosen rows so the exact sample can be reproduced and
checked later.

The selected rows are copied into memory as `float32` (32-bit numbers). This halves the memory needed
compared with the package's native 64-bit values and does not noticeably change tree-based results.

In [ ]:
t0 = time.perf_counter()
labels_all = np.asarray(y_train)
full_train = FRACTION == 1
if full_train:
    idx = np.arange(len(labels_all), dtype=np.int64)
else:
    idx = stratified_indices(labels_all, FRACTION, SEED_SAMPLE)
actual_population = selected_population(train_metadata, idx)
idx_hash = hashlib.sha256(idx.tobytes()).hexdigest()
X = gather_float32(x_train, idx)
y = labels_all[idx]
t_load = time.perf_counter() - t0

print(f"Rows selected: {len(idx):,} of {len(labels_all):,}")
display(pd.Series({"benign": int((y == 0).sum()), "malicious": int((y == 1).sum())},
                  name="class counts").to_frame())

## 9. Train the Random Forest

`RandomForestClassifier` is built from the settings chosen earlier, and `.fit(X, y)` trains it.
Setting `random_state` to the model seed makes the forest identical every time the same settings are run.
`n_jobs` lets several trees train at once across `JOBS` CPU threads.

No class weighting is applied, so the model treats both classes as they appear in the sample. The check
after training confirms the classes are ordered `[0, 1]`, which keeps the probability column we read
(`[:, 1]`) pointing at "malicious".

In [ ]:
model = RandomForestClassifier(
    n_estimators=TREES,
    n_jobs=JOBS,
    random_state=SEED_MODEL,
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=MAX_FEATURES,
)
t0 = time.perf_counter()
model.fit(X, y)
t_fit = time.perf_counter() - t0
assert list(model.classes_) == [0, 1]
print(f"Training finished in {t_fit:.1f} s")

## 10. Evaluate on the VALIDATION split

VALIDATION rows were never used for training, so they show how well the model handles unseen traffic.
The model scores them in chunks of `CHUNK_SIZE` rows to limit memory use. Each score is the model's
probability that a row is malicious, and `binary_metrics` converts those probabilities into counts and
scores using the `0.5` threshold.

In [ ]:
t0 = time.perf_counter()
y_true, proba = predict_validation(model, package, CHUNK_SIZE)
metrics = binary_metrics(y_true, proba)
t_eval = time.perf_counter() - t0
print(f"Scored {len(y_true):,} VALIDATION rows in {t_eval:.1f} s")

## 11. Review the results

The table lists the main scores for each class:

- **Precision**: of the rows flagged as this class, the share that truly belong to it.
- **Recall**: of the rows that truly belong to this class, the share that were found.
- **F1**: a single score that balances precision and recall.

Below it, **macro F1** is the average of the two classes' F1 scores, so benign and malicious traffic count
equally. **FPR** (false positive rate) is the share of benign rows wrongly flagged as malicious, which
matters because false alarms cost an administrator time. The confusion matrix shows the raw counts.

In [ ]:
per_class = pd.DataFrame(metrics["per_class"]).T[["precision", "recall", "f1", "support"]]
display(per_class)
print(f"Accuracy: {metrics['accuracy']:.6f}   Macro F1: {metrics['macro_f1']:.6f}   "
      f"FPR: {metrics['fpr']:.6f}")

cm = np.array([[metrics["TN"], metrics["FP"]], [metrics["FN"], metrics["TP"]]])
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1], ["benign", "malicious"])
ax.set_yticks([0, 1], ["benign", "malicious"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("VALIDATION confusion matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.show()

## 12. Assemble the result record

Everything needed to understand and repeat the run goes into one record: the model settings, the seeds,
the package version and fingerprints, the sampled rows, and the validation metrics. Hashes tie the result
to the exact data it came from. The run is marked `test_partition_used: false` because TEST was never
touched.

In [ ]:
parameters = {
    "algorithm": "RandomForestClassifier",
    "n_estimators": TREES,
    "class_weight": None,
    "random_state": SEED_MODEL,
    "max_depth": MAX_DEPTH,
    "min_samples_leaf": MIN_SAMPLES_LEAF,
    "max_features": MAX_FEATURES,
    "fit_dtype": "float32",
    "train_sampling": {
        "method": ("none: all TRAIN rows" if full_train
                   else "random sample inside TRAIN, same fraction per class, seeded, TRAIN only"),
        "fraction": FRACTION,
        "seed": None if full_train else SEED_SAMPLE,
        "rows": int(len(idx)),
        "rows_in_train": int(manifest["counts"]["TRAIN"]),
        "class_counts": {"benign": int((y == 0).sum()), "malicious": int((y == 1).sum())},
        "indices_sha256": idx_hash,
    },
}

result = {
    "run_id": RUN_ID,
    "package_version": manifest["package_version"],
    "package_manifest_sha256": package_verification["manifest_sha256"],
    "resolved_configuration": {"seeds": SEEDS, "parameters": parameters,
                               "input_view": "raw", "threshold": 0.5},
    "metrics": metrics,
    "train_population_sha256": actual_population["train_population_sha256"],
    "validation_population_sha256": manifest["files"]["VALIDATION_metadata.npy"]["sha256"],
    "parent_train_population_sha256": manifest["files"]["TRAIN_metadata.npy"]["sha256"],
    "actual_train_membership": actual_population,
}

## 13. Save the outputs

The run folder under `outputs/` receives:

- `model.joblib`: the trained model.
- `result.json`: the full result record.
- `train_sample_indices.npy`: which TRAIN rows were used.
- `package_verification.json`: proof that the dataset passed its checks.
- `run_notes.json`: timings, model size, memory use, and library versions.

In [ ]:
RUN_DIR.mkdir(parents=True, exist_ok=True)
np.save(RUN_DIR / "train_sample_indices.npy", idx)
(RUN_DIR / "package_verification.json").write_text(json.dumps(package_verification, indent=2))
joblib.dump(model, RUN_DIR / "model.joblib")
(RUN_DIR / "result.json").write_text(json.dumps(result, indent=2))

notes = {
    "run_id": RUN_ID,
    "validation_rows": int(manifest["counts"]["VALIDATION"]),
    "resources": {
        "load_sample_seconds": round(t_load, 1),
        "fit_seconds": round(t_fit, 1),
        "validation_predict_seconds": round(t_eval, 1),
        "model_size_mb": round((RUN_DIR / "model.joblib").stat().st_size / 2**20, 1),
        "peak_memory": peak_memory_mb(),
    },
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "test_partition_used": False,
}
(RUN_DIR / "run_notes.json").write_text(json.dumps(notes, indent=2))

mal = metrics["per_class"]["malicious"]
print(json.dumps({
    "rows_sampled": len(idx),
    "TN_FP_FN_TP": [metrics["TN"], metrics["FP"], metrics["FN"], metrics["TP"]],
    "macro_f1": metrics["macro_f1"],
    "fpr": metrics["fpr"],
    "malicious_recall": mal["recall"],
    "resources": notes["resources"],
}, indent=2))
print("\nSaved to", RUN_DIR.resolve())